In [1]:
from transformers import ViTForImageClassification, ViTImageProcessor
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import random_split
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
from torch.utils.data import Dataset

import os
for dirname, _, filenames in os.walk('.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
path = Path(os.path.join('..', 'input', 'sgfood-train-test', 'datasets'))
path_train = path/'train'
path_test = path/'test'

!ls {path}
!ls {path_train}

test  train
 Apple		     'Fish and chips'		   Papaya
 Apricot	     'Fish Ball Noodles'	  'pasta - red sauce'
'ayam penyet'	     'fish head curry'		   pear
'bak kut teh'	     'fried chicken'		  'pineapple tarts'
'bak kwa'	     'goreng pisang'		   popiah
 banana		     'green leafy vegetables'	   Porridge
'Ban Mian'	     'har cheong gai'		  'Prawn Noodle'
'bee hoon'	     'hokkien prawn mee'	  'rice dumpling'
 Bibimbap	     'Hor Fun'			  'roasted chicken'
 Blackberry	     'ice kacang'		   salad
'black pepper crab'  'Indian Prata'		  'salmon - grilled'
 blueberries	     'kebab - chicken'		  'sambal stingray'
 Burger		     'Kway Teow'		   sandwich
'cheese fries'	      Laksa			  'satay bee hoon'
'chicken rice'	     'Lor mee'			  'Seafood Noodles Soup'
'chicken soup'	     'Mee rebus'		  'siew mai'
'chilli crab'	     'Mee siam'			  'sirloin steak'
'Chinese fritters'    milk			  'Soft boiled eggs'
'chwee kueh'	     'Miso ramen, with fishcake'  'steamed grouper'
'Claypot Rice'	     'mixed vegetab

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model_name = "google/vit-base-patch16-224"
model = ViTForImageClassification.from_pretrained(model_name, image_size=160, num_labels=78, ignore_mismatched_sizes=True).to(device)
processor = ViTImageProcessor.from_pretrained(model_name)

# new_dropout_rate = 0.2
# model.config.hidden_dropout_prob = new_dropout_rate
# model.config.attention_probs_dropout_prob = new_dropout_rate

# model = ViTForImageClassification(config=model.config)

# for param in model.vit.parameters():
#     param.requires_grad = False

print(f"Hidden dropout: {model.config.hidden_dropout_prob}")
print(f"Attention dropout: {model.config.attention_probs_dropout_prob}")

model = torch.nn.DataParallel(model)
model.to(device)

preprocess_train = Compose([
    Resize((160, 160)),
    # transforms.RandomResizedCrop(size=160),
    # transforms.RandomHorizontalFlip(p=0.1),
    # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

preprocess_test = Compose([
    Resize((160, 160)),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

cuda


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([78]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([78, 768]) in the model instantiated
- vit.embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 101, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Hidden dropout: 0.0
Attention dropout: 0.0


In [4]:
full_dataset = ImageFolder(root=path_train, transform=preprocess_train)

validation_ratio = 0.2
num_valid = int(len(full_dataset) * validation_ratio)
num_train = len(full_dataset) - num_valid

train_dataset, valid_dataset = random_split(full_dataset, [num_train, num_valid])

train_loader = DataLoader(train_dataset, batch_size=150, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=150, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.003)
# optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-4)
# scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
criterion = torch.nn.CrossEntropyLoss()

In [5]:
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_accuracy):
        if self.best_score is None or val_accuracy > self.best_score:
            self.best_score = val_accuracy
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [6]:
def evaluate_model(model, data_loader):
    model.eval()
    running_loss = 0.0
    total = 0
    correct = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(pixel_values=inputs).logits

            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0.0)
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    return running_loss / len(data_loader), accuracy * 100, precision, recall, f1

In [7]:
def train_model(model, train_loader, valid_loader, epochs=30):
    early_stopping = EarlyStopping(patience=3)
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        total = 0
        correct = 0

        with tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}", unit="batch") as tepoch:
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
    
                optimizer.zero_grad()
                outputs = model(pixel_values=inputs).logits
    
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
    
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total

        valid_loss, valid_accuracy, _, _, _ = evaluate_model(model, valid_loader)

        # scheduler.step(valid_loss)

        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Train Loss: {running_loss/len(train_loader):.4f}, Accuracy: {train_accuracy:.2f}%")
        print(f"Valid Loss: {valid_loss:.4f}, Accuracy: {valid_accuracy:.2f}%")
        print("-" * 40)

        early_stopping(valid_accuracy)
        if early_stopping.early_stop:
            print("Training stopped because of no increase in validation accuracy.")
            break

In [8]:
train_model(model, train_loader, valid_loader, epochs=30)

Epoch 1/30:   0%|          | 0/165 [06:43<?, ?batch/s]


Epoch 1/30
Train Loss: 4.2412, Accuracy: 3.12%
Valid Loss: 4.0233, Accuracy: 5.32%
----------------------------------------


Epoch 2/30:   0%|          | 0/165 [05:50<?, ?batch/s]


Epoch 2/30
Train Loss: 3.7534, Accuracy: 9.88%
Valid Loss: 3.5286, Accuracy: 13.31%
----------------------------------------


Epoch 3/30:   0%|          | 0/165 [05:48<?, ?batch/s]


...

Epoch 16/30
Train Loss: 1.6523, Accuracy: 53.86%
Valid Loss: 2.8757, Accuracy: 32.15%
----------------------------------------


Epoch 17/30:   0%|          | 0/165 [06:06<?, ?batch/s]


Epoch 17/30
Train Loss: 1.5290, Accuracy: 57.04%
Valid Loss: 2.7984, Accuracy: 34.38%
----------------------------------------


Epoch 18/30:   0%|          | 0/165 [05:59<?, ?batch/s]


Epoch 18/30
Train Loss: 1.3895, Accuracy: 60.12%
Valid Loss: 2.8255, Accuracy: 34.54%
----------------------------------------
Training stopped because of no increase in validation accuracy.


In [9]:
def save_model(model, filepath):
    torch.save(model.state_dict(), filepath)
    print(f"Model saved to {filepath}")

save_model(model, '/kaggle/working/image_to_nutri_grade_transformer_4.pth')

Model saved to /kaggle/working/image_to_nutri_grade_transformer_4.pth


In [10]:
test_dataset = ImageFolder(path_test, transform=preprocess_test)
test_loader = DataLoader(test_dataset, batch_size=150, shuffle=False)
loss, accuracy, precision, recall, f1 = evaluate_model(model, test_loader)

print(f"Test Accuracy: {accuracy/100:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")

Test Accuracy: 0.3583
Test Precision: 0.3794
Test Recall: 0.3583
Test F1 Score: 0.3520


In [11]:
category_to_nutri_grade = {
    'Apple': 'A',
    'Apricot': 'A',
    'banana': 'A',
    'Blackberry': 'A',
    'blueberries': 'A',
    'Papaya': 'A',
    'orange': 'A',
    'pear': 'A',
    'salad': 'A',
    'mixed vegetables': 'A',
    'green leafy vegetables': 'A',
    'sandwich': 'A',
    'salmon - grilled': 'A',
    'Soft boiled eggs': 'A',
    'milk': 'A',
    'Nuts': 'A',
    'whole grain bread': 'A',
    'whole oats': 'A',
    'cooked brown rice': 'A',
    'cooked white rice': 'A',
    'corn': 'A',
    'Porridge': 'A',
    'yogurt': 'A',
    'thunder tea rice': 'A',

    'steamed grouper': 'B',
    'Ban Mian': 'B',
    'bee hoon': 'B',
    'Udon': 'B',
    'Fish Ball Noodles': 'B',
    'Seafood Noodles Soup': 'B',
    'Prawn Noodle': 'B',
    'sirloin steak': 'B',
    'pasta - red sauce': 'B',
    'dumpling': 'B',
    'siew mai': 'B',
    'Bibimbap': 'B',
    'chicken soup': 'B',
    'muesli': 'B',
    'popiah': 'B',
    'kebab - chicken': 'B',
    'sushi': 'B',
    'roasted chicken': 'B',
    'otak': 'B',

    'Lor mee': 'C',
    'Mee rebus': 'C',
    'Mee siam': 'C',
    'nasi lemak': 'C',
    'bak kut teh': 'C',
    'Duck Rice': 'C',
    'Claypot Rice': 'C',
    'rice dumpling': 'C',
    'pineapple tarts': 'C',
    'Miso ramen, with fishcake': 'C',
    'chwee kueh': 'C',
    'chicken rice': 'C',
    'Hor Fun': 'C',
    'hokkien prawn mee': 'C',
    'goreng pisang': 'C',
    'tacos and nachos': 'C',

    'Burger': 'D',
    'sambal stingray': 'D',
    'oyster omelette': 'D',
    'cheese fries': 'D',
    'bak kwa': 'D',
    'chilli crab': 'D',
    'black pepper crab': 'D',
    'fish head curry': 'D',
    'Indian Prata': 'D',
    'ayam penyet': 'D',
    'Kway Teow': 'D',
    'Fish and chips': 'D',
    'fried chicken': 'D',
    'har cheong gai': 'D',
    'satay bee hoon': 'D',
    'ice kacang': 'D',
    'Laksa': 'D',
    'Chinese fritters': 'D',
    'curry puff': 'D'
}

nutri_grade_to_numeric = {
    'A': 0,
    'B': 1,
    'C': 2,
    'D': 3
}

In [12]:
def evaluate_model_nutri_grade(model, data_loader, idx_to_category):
    model.eval()
    running_loss = 0.0
    total = 0
    correct = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(pixel_values=inputs).logits

            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            pred_grades = [category_to_nutri_grade[idx_to_category[i.item()]] for i in preds.cpu()]
            label_grades = [category_to_nutri_grade[idx_to_category[i.item()]] for i in labels.cpu()]
            correct += sum([p == l for p, l in zip(pred_grades, label_grades)])
            all_preds.extend(pred_grades)
            all_labels.extend(label_grades)

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0.0)
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    return running_loss / len(data_loader), accuracy * 100, precision, recall, f1

In [20]:
model_name = "google/vit-base-patch16-224"
model = ViTForImageClassification.from_pretrained(model_name, image_size=160, num_labels=78, ignore_mismatched_sizes=True).to(device)
# model.load_state_dict(torch.load("/kaggle/input/transformer-1/pytorch/default/1/image_to_nutri_grade_transformer.pth"))
model = torch.nn.DataParallel(model)
model.load_state_dict(torch.load("/kaggle/working/image_to_nutri_grade_transformer_4.pth"))
model = model.to(device)

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([78]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([78, 768]) in the model instantiated
- vit.embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 101, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-20-51e6e61d4e89>:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/py

In [21]:
test_dataset = ImageFolder(path_test, transform=preprocess_test)
test_loader = DataLoader(test_dataset, batch_size=150, shuffle=False)
idx_to_category = {v: k for k, v in test_dataset.class_to_idx.items()}
loss, accuracy, precision, recall, f1 = evaluate_model_nutri_grade(model, test_loader, idx_to_category)

print(f"Test Accuracy: {accuracy/100:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")

Test Accuracy: 0.5846
Test Precision: 0.5858
Test Recall: 0.5846
Test F1 Score: 0.5846
